# TinyGPT

A minimal GPT-style language model implemented from scratch using PyTorch.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Constants

| Constant | Description |
|--------|-------------|
| `batch_size` | Number of sequences processed at once during training |
| `block_size` | Context length (how many tokens the model can look at) |
| `max_iters` | Total number of training iterations |
| `eval_interval` | How often to print training loss |
| `learning_rate` | Step size for optimizer updates |
| `n_embd` | Embedding dimension (size of token vectors) |
| `n_head` | Number of attention heads |
| `n_layer` | Number of transformer blocks |
| `dropout` | Dropout rate used for regularization |
| `device` | Training device: `"cuda"` if GPU available, otherwise `"cpu"` |

In [2]:
batch_size = 32
block_size = 256
max_iters = 20000
learning_rate = 3e-4
eval_interval = 200
device = "cuda" if torch.cuda.is_available() else "cpu"
amp_dtype = torch.bfloat16 if device == "cuda" else torch.float32
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.2

In [3]:
with open("data.txt", "r", encoding="utf-8") as f:
    corpus = f.read()

unique_characters = sorted(set(corpus))
# print(unique_characters)
vocabulary_size = len(unique_characters)

In [4]:
char_to_index = {character: index for index, character in enumerate(unique_characters)}
index_to_char = {index: character for index, character in enumerate(unique_characters)}

encode = lambda text: [char_to_index[character] for character in text]
decode = lambda indices: "".join([index_to_char[index] for index in indices])

In [5]:
data = torch.tensor(encode(corpus), dtype=torch.long)
print(data.shape)

torch.Size([1115394])


In [6]:
N = 0.9
train_data_size = int(N * len(data))

train_data = data[:train_data_size].to(device)
val_data = data[train_data_size:].to(device)

In [7]:
def get_batch(split):
    source = train_data if split == "train" else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in starts])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in starts])
    return x, y

x, y = get_batch("train")
print(x.shape, y.shape)

torch.Size([32, 256]) torch.Size([32, 256])


In [8]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        loss = None

        if targets is not None:
            batch, time, channels = logits.shape
            loss = F.cross_entropy(
                logits.view(batch * time, channels),
                targets.view(batch * time),
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            context = idx[:, -block_size:]
            logits, _ = self(context)
            probabilities = F.softmax(logits[:, -1, :], dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1)
            idx = torch.cat((idx, next_token), dim=1)
        return idx

model = BigramLanguageModel(vocabulary_size).to(device)
logits, loss = model(x, y)
print(logits.shape, loss.item())

torch.Size([32, 256, 65]) 4.560153961181641


In [9]:
print("Vocabulary size:", vocabulary_size)
print("Dataset: ", len(corpus))

Vocabulary size: 65
Dataset:  1115394


In [10]:
@torch.no_grad()
def estimate_loss():
    results = {}
    model.eval()

    for split in ("train", "val"):
        losses = []

        for _ in range(50):
            batch_x, batch_y = get_batch(split)
            with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
                _, batch_loss = model(batch_x, batch_y)
            losses.append(batch_loss.item())

        results[split] = sum(losses) / len(losses)

    model.train()
    return results


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    fused=(device == "cuda"),
)

import os

bigram_checkpoint_path = "bigram_checkpoint.pt"
start_step = 0

if os.path.exists(bigram_checkpoint_path):
    ckpt = torch.load(bigram_checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_step = ckpt["step"] + 1
    print(f"Resuming from step {start_step}")
else:
    print("No checkpoint found, starting from scratch")

for step in range(start_step, max_iters):
    if step % eval_interval == 0:
        print(f"step {step}: {estimate_loss()}")
        torch.save({
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "step": step,
        }, bigram_checkpoint_path)

    batch_x, batch_y = get_batch("train")

    with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
        _, loss = model(batch_x, batch_y)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"step {max_iters}: {estimate_loss()}")

torch.save({
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "step": max_iters - 1,
}, bigram_checkpoint_path)

prompt = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device,
)

sample = model.generate(
    prompt,
    max_new_tokens=500,
)

print(decode(sample[0].tolist()))

Resuming from step 20000
step 20000: {'train': 2.4600477504730223, 'val': 2.488774199485779}

WAMAUThe yo arin ler IO:
VINond
VOMianerdiceereller cesil onon f y?
Y:
Ande.
QUCat ounthiouccepoftherveall'lswbof hankeschea tiorilar o hos t pe bly bryooucee r.
Sen II co'll se?
NBue wo meno helalourerare, and m gont agit cu ses s h,
My imes sh titinn hithye oumushe wheood

Gis ay, nod buco relluntol LI wous a l bee, thar hie inthachtousLONaindind wof
W!

Lain hint t ildind pr ofur he, machicour
IO:
F llow, llet anik o ss. sty t theancoto the f taread'FLUChe ste ve sung kndey?
NELarchinor'thDUL


In [11]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        _, time, _ = x.shape
        key = self.key(x)
        query = self.query(x)
        weights = query @ key.transpose(-2, -1) * key.shape[-1] ** -0.5
        weights = weights.masked_fill(
            self.mask[:time, :time] == 0,
            float("-inf"),
        )
        weights = self.dropout(F.softmax(weights, dim=-1))
        return weights @ self.value(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )
        self.projection = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        combined = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.projection(combined))


class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.network(x)


class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.attention = MultiHeadAttention(n_head, head_size)
        self.feed_forward = FeedForward()
        self.norm1 = nn.LayerNorm(n_embd)
        self.norm2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        assert n_embd % n_head == 0
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[TransformerBlock() for _ in range(n_layer)]
        )
        self.final_norm = nn.LayerNorm(n_embd)
        self.language_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        _, time = idx.shape
        positions = torch.arange(time, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        logits = self.language_head(self.final_norm(self.blocks(x)))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                targets.reshape(-1),
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        was_training = self.training
        self.eval()

        for _ in range(max_new_tokens):
            context = idx[:, -block_size:]
            logits, _ = self(context)
            probabilities = F.softmax(logits[:, -1, :], dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1)
            idx = torch.cat((idx, next_token), dim=1)

        if was_training:
            self.train()

        return idx


model = TinyGPT(vocabulary_size).to(device)
logits, loss = model(x, y)
print("Logits shape:", logits.shape)
print("Initial loss:", loss.item())

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, fused=(device == "cuda"))

Logits shape: torch.Size([32, 256, 65])
Initial loss: 4.319011688232422


In [12]:
import os

start_step = 0

if os.path.exists("checkpoint.pt"):
    ckpt = torch.load("checkpoint.pt", map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_step = ckpt["step"] + 1
    print(f"Resuming from step {start_step}")
else:
    print("No checkpoint found, starting from scratch")

Resuming from step 18601


In [13]:
for step in range(start_step, max_iters):
    if step % eval_interval == 0:
        print(f"step {step}: {estimate_loss()}")
        torch.save({
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "step": step,
        }, "checkpoint.pt")

    batch_x, batch_y = get_batch("train")

    with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == "cuda")):
        _, loss = model(batch_x, batch_y)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"step {max_iters}: {estimate_loss()}")

torch.save({
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "step": max_iters - 1,
}, "checkpoint.pt")

prompts = [
    "ROMEO:",
    "KING:",
    "First Citizen:",
    "To be",
]

for prompt_text in prompts:
    prompt = torch.tensor(
        [encode(prompt_text)],
        dtype=torch.long,
        device=device,
    )

    sample = model.generate(prompt, max_new_tokens=500)
    print("=" * 80)
    print(decode(sample[0].tolist()))

step 18800: {'train': 1.3197594094276428, 'val': 1.548291757106781}
step 19000: {'train': 1.3172939944267272, 'val': 1.5477762484550477}
step 19200: {'train': 1.3099971842765807, 'val': 1.54726722240448}
step 19400: {'train': 1.3204731297492982, 'val': 1.5534697341918946}
step 19600: {'train': 1.314811110496521, 'val': 1.5549244809150695}
step 19800: {'train': 1.3150288367271423, 'val': 1.550408411026001}
step 20000: {'train': 1.308550684452057, 'val': 1.5394506883621215}
ROMEO:
Well, my name counted his body.
Juliet wantil; by mew's thy still? O lock.

LADY CAPULET:
For but I'll displice.

POMPEY:
No, I am thee they men:
'Tis my is prey not please
That say words I tell not do me and her,
That's the false on enemies?

LEONTES:
I think upon my mother; persuade, and Coriolanus:
You have it.

OF HENRY VI

HENRY VOLINGBROKE:
Baspassed you many that new barr'd in so sovereign'd:
Ay, and York man more love, and thou hast--
I am the fardmen of Gland's sosters.

CLIFFORD:
What
KING:
That mark'

In [14]:
torch.save(model.state_dict(), 'tinygpt.pt')